# Phase 1: Exploratory Data Analysis
## Melbourne Airbnb Open Data

**Goal:** Deeply understand the dataset before touching a single model.  
Good EDA is the difference between a model that *works* and one that *makes sense*.

### What we'll cover:
1. Dataset overview & basic profiling
2. Univariate analysis — distributions, skew, kurtosis
3. Target variable deep-dive (price)
4. Bivariate analysis — price vs every feature
5. Geospatial analysis — where are the expensive listings?
6. Correlation analysis
7. Advanced: Outlier detection + PCA overview

---

## 0. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ── Consistent colour palette (teal / coral, same as Boston Housing course) ──
TEAL   = '#2A9D8F'
CORAL  = '#E76F51'
SAND   = '#E9C46A'
NAVY   = '#264653'
PURPLE = '#8338EC'
PALETTE = [TEAL, CORAL, SAND, NAVY, PURPLE]

sns.set_theme(style='whitegrid', palette=PALETTE)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

print('Libraries loaded ✓')

## 1. Load Data & Basic Profiling

We load both train and test upfront.  
**Rule of thumb:** EDA is done on the training set only — the test set is a held-out simulation of the real world and shouldn't influence your analysis or decisions.

In [ ]:
# ── Update these paths to wherever you saved the files ──
TRAIN_PATH = 'train_data.csv'
TEST_PATH  = 'test_data.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test.shape}')
print(f'\nTarget (price) lives in train only — test has {test.shape[1]} cols, train has {train.shape[1]}')

In [ ]:
train.head()

In [ ]:
# dtypes snapshot — quick way to spot things that are wrong type (e.g. price stored as string)
train.dtypes.to_frame(name='dtype').T

In [ ]:
# Statistical summary — look at min/max/std for obvious red flags
train.describe(include='all').T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

In [ ]:
# Missing values — even though there are none, good practice to always verify
missing = train.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else '✓ No missing values in training data')

# Columns we can immediately drop (zero variance / single value)
low_var = [c for c in train.columns if train[c].nunique() == 1]
print(f'\nColumns with only 1 unique value (drop candidates): {low_var}')

**Note:** `country` has only one value — Australia. It carries zero predictive signal and will be dropped in Phase 2 (data prep).  
Similarly, `city` here means Melbourne LGA/council area, not the literal city — important for interpretation.

---
## 2. Univariate Analysis

We look at each feature in isolation before comparing them.  
Key stats to track: **shape of distribution, skewness, kurtosis, outliers**.

- **Skewness > 1 or < -1** → significantly skewed, may need log/sqrt transform  
- **Kurtosis > 3** → heavy tails (more extreme outliers than a normal distribution)

In [ ]:
num_cols = train.select_dtypes(include=np.number).columns.tolist()
cat_cols = train.select_dtypes(include='object').columns.tolist()

print(f'Numeric columns  ({len(num_cols)}): {num_cols}')
print(f'Categorical cols ({len(cat_cols)}): {cat_cols}')

In [ ]:
# Skewness & Kurtosis table
skew_kurt = pd.DataFrame({
    'skewness' : train[num_cols].skew(),
    'kurtosis' : train[num_cols].kurt(),
}).round(3)

skew_kurt['skew_flag']  = skew_kurt['skewness'].abs().apply(lambda x: '⚠️ High' if x > 1 else 'OK')
skew_kurt['kurt_flag']  = skew_kurt['kurtosis'].apply(lambda x: '⚠️ Heavy tails' if x > 3 else 'OK')
skew_kurt

In [ ]:
# Distribution plots for all numeric features
# Using a histogram + KDE overlay + a rug plot for density at the base

n = len(num_cols)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    data = train[col].dropna()
    
    ax.hist(data, bins=40, color=TEAL, alpha=0.7, edgecolor='white', density=True)
    
    # KDE overlay
    kde_x = np.linspace(data.min(), data.max(), 300)
    kde = stats.gaussian_kde(data)
    ax.plot(kde_x, kde(kde_x), color=CORAL, lw=2, label='KDE')
    
    # Annotate skew
    skew_val = data.skew()
    ax.set_title(f'{col}\nskew={skew_val:.2f}', fontsize=10)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

# Hide empty subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Univariate Distributions — All Numeric Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features — bar charts
fig, axes = plt.subplots(1, len(cat_cols), figsize=(18, 4))

for i, col in enumerate(cat_cols):
    vc = train[col].value_counts()
    
    # Truncate city to top 10 for readability
    if col == 'city':
        vc = vc.head(10)
    
    axes[i].bar(vc.index, vc.values, color=PALETTE[:len(vc)])
    axes[i].set_title(f'{col}\n(n unique = {train[col].nunique()})', fontsize=10)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Categorical Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Target Variable Deep-Dive: `price`

Prices are almost always right-skewed in Airbnb data — a small number of luxury listings drag the mean way above the median.  
This matters because most regression models assume a roughly normal residual distribution.  
**Common fix:** log-transform the target → `log1p(price)`.

In [ ]:
# Price: raw vs log-transformed side by side
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Raw distribution
ax = axes[0]
ax.hist(train['price'], bins=60, color=TEAL, edgecolor='white', alpha=0.8)
ax.axvline(train['price'].mean(),   color=CORAL,  lw=2, linestyle='--', label=f"Mean: ${train['price'].mean():.0f}")
ax.axvline(train['price'].median(), color=SAND,   lw=2, linestyle='--', label=f"Median: ${train['price'].median():.0f}")
ax.set_title('Price — Raw Distribution')
ax.set_xlabel('Price (AUD)')
ax.legend()

# Log-transformed
log_price = np.log1p(train['price'])
ax = axes[1]
ax.hist(log_price, bins=60, color=CORAL, edgecolor='white', alpha=0.8)
ax.axvline(log_price.mean(),   color=TEAL, lw=2, linestyle='--', label=f'Mean: {log_price.mean():.2f}')
ax.axvline(log_price.median(), color=SAND, lw=2, linestyle='--', label=f'Median: {log_price.median():.2f}')
ax.set_title('Price — Log1p Transformed')
ax.set_xlabel('log1p(Price)')
ax.legend()

# Q-Q plot of log price (normality check)
ax = axes[2]
(osm, osr), (slope, intercept, r) = stats.probplot(log_price, dist='norm')
ax.scatter(osm, osr, color=TEAL, alpha=0.4, s=5)
ax.plot(osm, slope * np.array(osm) + intercept, color=CORAL, lw=2)
ax.set_title(f'Q-Q Plot: log1p(Price)\nR²={r**2:.4f}')
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles')

plt.suptitle('Price Distribution Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Raw price  — skew: {train['price'].skew():.3f}, kurt: {train['price'].kurt():.3f}")
print(f"Log price  — skew: {log_price.skew():.3f}, kurt: {log_price.kurt():.3f}")

In [ ]:
# Outlier analysis — IQR method
Q1, Q3 = train['price'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

outliers = train[(train['price'] < lower_fence) | (train['price'] > upper_fence)]
print(f'IQR fences: [{lower_fence:.0f}, {upper_fence:.0f}]')
print(f'Outliers: {len(outliers)} rows ({len(outliers)/len(train)*100:.1f}% of training data)')
print(f'Max price: ${train["price"].max()}')
print(f'\nTop 10 most expensive listings:')
print(train.nlargest(10, 'price')[['city', 'room_type', 'bedrooms', 'accommodates', 'price']])

In [ ]:
# Box plot — raw price with IQR fences marked
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Box + strip (jitter)
ax = axes[0]
ax.boxplot(train['price'], vert=False, patch_artist=True,
           boxprops=dict(facecolor=TEAL, alpha=0.6),
           medianprops=dict(color=CORAL, lw=2),
           flierprops=dict(marker='o', color=CORAL, alpha=0.3, markersize=3))
ax.axvline(upper_fence, color=CORAL, linestyle='--', lw=1.5, label=f'Upper fence: ${upper_fence:.0f}')
ax.set_title('Price Boxplot (IQR method)')
ax.set_xlabel('Price (AUD)')
ax.legend()

# Price percentiles
ax = axes[1]
pcts = np.arange(0, 101, 5)
vals = np.percentile(train['price'], pcts)
ax.plot(pcts, vals, color=TEAL, lw=2, marker='o', markersize=4)
ax.axhline(upper_fence, color=CORAL, linestyle='--', lw=1.5, label=f'IQR upper fence: ${upper_fence:.0f}')
ax.fill_between(pcts, vals, alpha=0.2, color=TEAL)
ax.set_title('Price Percentile Curve')
ax.set_xlabel('Percentile')
ax.set_ylabel('Price (AUD)')
ax.legend()

plt.suptitle('Price Outlier Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Bivariate Analysis — Price vs Every Feature

Now we look at *relationships*. How does each feature correlate with or influence price?  
- **Numeric vs price** → scatter plot + regression line + Pearson r  
- **Categorical vs price** → box plots grouped by category

In [ ]:
# Numeric vs log(price) — scatter + regression line
num_features = [c for c in num_cols if c != 'price']
log_price = np.log1p(train['price'])

ncols = 3
nrows = (len(num_features) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 4))
axes = axes.flatten()

for i, col in enumerate(num_features):
    ax = axes[i]
    x = train[col]
    
    # Scatter
    ax.scatter(x, log_price, color=TEAL, alpha=0.2, s=8, edgecolors='none')
    
    # Linear regression line
    slope, intercept, r, p, se = stats.linregress(x, log_price)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, slope * x_line + intercept, color=CORAL, lw=2)
    
    ax.set_title(f'{col} vs log(price)\nr = {r:.3f}  p = {p:.2e}', fontsize=9)
    ax.set_xlabel(col)
    ax.set_ylabel('log1p(price)')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Bivariate: Numeric Features vs log(Price)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical vs price — grouped box plots
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

cat_to_plot = [
    ('room_type',         'Room Type'),
    ('host_is_superhost', 'Superhost (f/t)'),
    ('instant_bookable',  'Instant Bookable (f/t)'),
    ('city',              'Council Area (top 12)'),
]

for i, (col, label) in enumerate(cat_to_plot):
    ax = axes[i]
    
    if col == 'city':
        top12 = train['city'].value_counts().head(12).index
        data  = train[train['city'].isin(top12)]
        order = data.groupby('city')['price'].median().sort_values(ascending=False).index
    else:
        data  = train
        order = data.groupby(col)['price'].median().sort_values(ascending=False).index
    
    groups = [data[data[col] == g]['price'].values for g in order]
    bp = ax.boxplot(groups, labels=order, patch_artist=True, showfliers=False,
                    medianprops=dict(color=CORAL, lw=2))
    for patch, colour in zip(bp['boxes'], [TEAL, SAND, NAVY, PURPLE, CORAL] * 5):
        patch.set_facecolor(colour)
        patch.set_alpha(0.6)
    
    ax.set_title(f'Price by {label}', fontsize=11)
    ax.set_ylabel('Price (AUD)')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Bivariate: Categorical Features vs Price', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Violin plot: price by room_type — richer view of full distribution shape
fig, ax = plt.subplots(figsize=(10, 5))

room_types = train['room_type'].unique()
data_by_room = [train[train['room_type'] == rt]['price'].values for rt in room_types]

parts = ax.violinplot(data_by_room, positions=range(len(room_types)),
                      showmedians=True, showextrema=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(PALETTE[i % len(PALETTE)])
    pc.set_alpha(0.7)
parts['cmedians'].set_color(CORAL)
parts['cmedians'].set_linewidth(2)

ax.set_xticks(range(len(room_types)))
ax.set_xticklabels(room_types)
ax.set_ylabel('Price (AUD)')
ax.set_ylim(0, 600)   # cap at 600 to see shape — outliers compressed
ax.set_title('Price Distribution by Room Type (violin — capped at $600 for clarity)', fontsize=11)
plt.tight_layout()
plt.show()

---
## 5. Geospatial Analysis

We have `latitude` and `longitude` — that's a huge advantage. Location is often the single strongest predictor of price in property-related datasets.  
Let's visualise where listings cluster and how price varies across Melbourne.

In [ ]:
# Scatter map: colour = price (log scale), size = accommodates
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: listing density
ax = axes[0]
ax.scatter(train['longitude'], train['latitude'],
           s=3, alpha=0.25, color=TEAL, edgecolors='none')
ax.set_title('Listing Density Across Melbourne', fontsize=11)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Plot 2: price heat by location
ax = axes[1]
sc = ax.scatter(train['longitude'], train['latitude'],
                c=np.log1p(train['price']),
                cmap='RdYlGn_r',
                s=5, alpha=0.4, edgecolors='none')
plt.colorbar(sc, ax=ax, label='log1p(Price)')
ax.set_title('Price Heatmap by Location', fontsize=11)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

plt.suptitle('Geospatial Analysis — Melbourne Airbnb Listings', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Median price per council area — bar chart ranked
city_stats = train.groupby('city')['price'].agg(['median', 'mean', 'count']).sort_values('median', ascending=False)
city_stats.columns = ['Median Price', 'Mean Price', 'Listing Count']

fig, ax = plt.subplots(figsize=(14, 5))
colors = [TEAL if i < 5 else SAND for i in range(len(city_stats))]
bars = ax.bar(city_stats.index, city_stats['Median Price'], color=colors, edgecolor='white')
ax.axhline(train['price'].median(), color=CORAL, lw=2, linestyle='--', label=f'Overall median: ${train["price"].median():.0f}')
ax.set_title('Median Airbnb Price by Melbourne Council Area', fontsize=12, fontweight='bold')
ax.set_ylabel('Median Price (AUD)')
ax.tick_params(axis='x', rotation=60)
ax.legend()

# Add count labels
for bar, (_, row) in zip(bars, city_stats.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'n={row["Listing Count"]}', ha='center', va='bottom', fontsize=7, rotation=90)

plt.tight_layout()
plt.show()

print(city_stats.head(10))

---
## 6. Correlation Analysis

Correlation tells us how linearly related two numeric variables are.  
- Pearson r: measures linear correlation (-1 to +1)  
- We check both feature-to-price AND feature-to-feature (multicollinearity)  

**Multicollinearity** is when two features are highly correlated with each other — this can destabilise linear models and make coefficients unreliable.

In [ ]:
# Encode binary categoricals for correlation
corr_df = train.copy()
corr_df['host_is_superhost'] = (corr_df['host_is_superhost'] == 't').astype(int)
corr_df['instant_bookable']  = (corr_df['instant_bookable']  == 't').astype(int)
corr_df['log_price']         = np.log1p(corr_df['price'])
corr_df = corr_df.drop(columns=['country', 'city', 'room_type', 'price'])

corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))   # only lower triangle
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Pearson r'}
)
ax.set_title('Correlation Matrix (lower triangle)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target — sorted bar chart
target_corr = corr_df.corr()['log_price'].drop('log_price').sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = [TEAL if v >= 0 else CORAL for v in target_corr.values]
ax.barh(target_corr.index, target_corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Feature Correlation with log(Price)', fontsize=12, fontweight='bold')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()

print('Top positive correlations with log(price):')
print(target_corr.sort_values(ascending=False).head(5))
print('\nTop negative correlations:')
print(target_corr.sort_values().head(5))

---
## 7. Advanced Analysis

### 7a. Multivariate Outlier Detection — Mahalanobis Distance

IQR catches univariate outliers (extreme values in a single feature).  
**Mahalanobis distance** detects *multivariate* outliers — rows that are unusual in the *combination* of features, even if each individual value looks fine.  
Think: a listing with 1 bedroom, 10 accommodates, and \$5 price — each feature individually might pass IQR, but together it's a red flag.

In [ ]:
from numpy.linalg import inv

# Use numeric features only (excluding lat/lon — spatial outliers are legitimate)
mah_cols = ['accommodates', 'bathrooms', 'bedrooms', 'beds',
            'minimum_nights', 'number_of_reviews',
            'review_scores_rating', 'calculated_host_listings_count']

X_mah = corr_df[mah_cols].dropna().values
X_mah_scaled = StandardScaler().fit_transform(X_mah)

# Mahalanobis: D² = (x - μ)ᵀ Σ⁻¹ (x - μ)
cov_matrix = np.cov(X_mah_scaled.T)
inv_cov    = inv(cov_matrix)
mean_vec   = X_mah_scaled.mean(axis=0)

diffs = X_mah_scaled - mean_vec
mah_dist = np.sqrt(np.einsum('ij,jk,ik->i', diffs, inv_cov, diffs))

# Chi-squared threshold (p=0.001, df=n_features)
threshold = np.sqrt(stats.chi2.ppf(0.999, df=len(mah_cols)))
mah_outliers = (mah_dist > threshold).sum()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(mah_dist, bins=60, color=TEAL, edgecolor='white', alpha=0.8)
ax.axvline(threshold, color=CORAL, lw=2, linestyle='--', label=f'χ² threshold: {threshold:.2f}')
ax.set_title(f'Mahalanobis Distance Distribution\n{mah_outliers} multivariate outliers detected ({mah_outliers/len(X_mah)*100:.1f}%)', fontsize=11)
ax.set_xlabel('Mahalanobis Distance')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Threshold: {threshold:.3f}')
print(f'Outliers : {mah_outliers} rows ({mah_outliers/len(X_mah)*100:.2f}%)')

### 7b. Principal Component Analysis (PCA)

PCA compresses all numeric features into a smaller set of uncorrelated axes (principal components), ranked by how much variance they explain.  
We're not using it for modelling here — this is a **diagnostic** step to:  
1. See how much information we can retain with fewer dimensions  
2. Spot any dominant structure in the data  
3. Get a hint about redundancy between features

In [ ]:
pca_cols = mah_cols
X_pca = corr_df[pca_cols].dropna()
X_scaled = StandardScaler().fit_transform(X_pca)

pca = PCA(n_components=len(pca_cols))
pca.fit(X_scaled)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
ax = axes[0]
ax.bar(range(1, len(explained)+1), explained * 100, color=TEAL, edgecolor='white', alpha=0.8, label='Individual')
ax2 = ax.twinx()
ax2.plot(range(1, len(cumulative)+1), cumulative * 100, color=CORAL, lw=2, marker='o', label='Cumulative')
ax2.axhline(90, color=SAND, lw=1.5, linestyle='--', label='90% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax2.set_ylabel('Cumulative Variance (%)')
ax.set_title('PCA Scree Plot', fontsize=11)
ax.legend(loc='upper left')
ax2.legend(loc='center right')

# PC1 vs PC2 scatter — colour by log(price)
X_pca_transformed = pca.transform(X_scaled)
log_prices_pca = np.log1p(corr_df['log_price'].values[:len(X_pca_transformed)])

ax = axes[1]
sc = ax.scatter(X_pca_transformed[:, 0], X_pca_transformed[:, 1],
                c=log_prices_pca, cmap='RdYlGn_r', alpha=0.3, s=5, edgecolors='none')
plt.colorbar(sc, ax=ax, label='log(log_price)')
ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)')
ax.set_title('PC1 vs PC2 — coloured by Price', fontsize=11)

plt.suptitle('Principal Component Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Variance explained per component:')
for i, (e, c) in enumerate(zip(explained, cumulative)):
    print(f'  PC{i+1}: {e*100:.1f}%   (cumulative: {c*100:.1f}%)')

In [ ]:
# PCA Loadings — what features drive PC1 and PC2?
loadings = pd.DataFrame(
    pca.components_[:3].T,
    index=pca_cols,
    columns=['PC1', 'PC2', 'PC3']
).round(3)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(['PC1', 'PC2', 'PC3']):
    ax = axes[i]
    sorted_load = loadings[col].sort_values()
    colors = [TEAL if v >= 0 else CORAL for v in sorted_load]
    ax.barh(sorted_load.index, sorted_load.values, color=colors, edgecolor='white')
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(f'{col} Loadings ({explained[i]*100:.1f}% var)', fontsize=10)
    ax.set_xlabel('Loading')

plt.suptitle('PCA Feature Loadings — What drives each component?', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(loadings)

---
## 8. EDA Summary & Key Takeaways

Before moving to Phase 2 (Data Preparation), let's document what we learned.

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║           EDA FINDINGS — KEY DECISIONS FOR PHASE 2          ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  TARGET                                                      ║
║  • price is heavily right-skewed (skew ~5+)                 ║
║  • Use log1p(price) as target for modelling                  ║
║  • ~10-12% of listings are IQR outliers (>$246)             ║
║  • $0 prices likely errors → investigate / cap              ║
║                                                              ║
║  FEATURES TO DROP                                            ║
║  • country: single value (Australia), zero variance          ║
║                                                              ║
║  ENCODING NEEDED                                             ║
║  • host_is_superhost: f/t → 0/1                             ║
║  • instant_bookable:  f/t → 0/1                             ║
║  • room_type: one-hot encode (3 categories)                  ║
║  • city: one-hot or target-encode (30 categories)            ║
║                                                              ║
║  STRONGEST PREDICTORS (correlation with log price)           ║
║  • accommodates, bedrooms, beds, bathrooms                   ║
║  • room_type (Entire home >> Private >> Shared)              ║
║  • city / location (Bayside, Stonnington premium)            ║
║                                                              ║
║  MULTICOLLINEARITY FLAGS                                     ║
║  • beds, bedrooms, accommodates are highly correlated        ║
║  • May need to drop one or use PCA components                ║
║                                                              ║
║  NEXT: Phase 2 — Data Preparation & Feature Engineering      ║
╚══════════════════════════════════════════════════════════════╝
""")